# Codificador Semantico
Ejecuta **todas las celdas** (Entorno de ejecucion -> Ejecutar todo) y abre la URL que aparece abajo.

**Requisito:** Cuenta Google para usar Colab.

In [ ]:
# Celda 1: Instalar dependencias
!pip install flask requests numpy scipy scikit-learn umap-learn hdbscan -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Ollama: instalar via script oficial (requiere zstd para .tar.zst nuevos)
!apt-get install -y -q zstd 2>&1 | tail -1
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3

import shutil
if not shutil.which('ollama'):
    print('Fallback: descargando binario Ollama directo (.tar.zst)...')
    !curl -fsSL https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst -o /tmp/ollama.tar.zst
    !tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C /usr/local/
    !chmod +x /usr/local/bin/ollama

assert shutil.which('ollama'), 'ERROR: Ollama no quedo instalado. Re-ejecuta esta celda.'
print('OK Dependencias instaladas (Flask, paso 7, cloudflared, Ollama en ' + shutil.which('ollama') + ')')


In [ ]:
# Celda 2: Configuracion
import os, zipfile, shutil, subprocess, time
import requests as req_dl

# Asegurar PATH y Ollama instalado (por si celda 1 fallo silencioso)
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
if not shutil.which('ollama'):
    print('Ollama no encontrado, instalando binario directo...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=False)
    subprocess.run(['curl', '-fsSL',
                    'https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst',
                    '-o', '/tmp/ollama.tar.zst'], check=True)
    subprocess.run(['tar', '--use-compress-program=unzstd', '-xf', '/tmp/ollama.tar.zst', '-C', '/usr/local/'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/ollama'], check=True)
    assert shutil.which('ollama'), 'ERROR: Ollama no se pudo instalar'
    print('OK Ollama instalado en ' + shutil.which('ollama'))

API_KEY = input('Ingresa tu API Key (Google AI Studio o OpenRouter): ').strip()
MODEL   = 'gemini-3.1-flash-lite-preview'
API_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/chat/completions'
PORT    = 8765

# Descargar dist/ desde GitHub Release publico (siempre latest)
DIST_ZIP = 'https://github.com/aacevedoc-ds/Cod_AA/releases/latest/download/dist.zip'
DIST_DIR = '/content/dist'

if not os.path.exists(DIST_DIR):
    print('Descargando app (latest release)...')
    r = req_dl.get(DIST_ZIP, timeout=60)
    r.raise_for_status()
    with open('/tmp/dist.zip', 'wb') as f:
        f.write(r.content)
    with zipfile.ZipFile('/tmp/dist.zip') as z:
        z.extractall('/content')
    print('OK App descargada en ' + DIST_DIR)
else:
    print('OK App ya disponible en ' + DIST_DIR)

# Descargar analytics.py desde el repo (logica Python del paso 7)
ANALYTICS_URL = 'https://raw.githubusercontent.com/aacevedoc-ds/Cod_AA/main/codificador-local/analytics.py'
print('Descargando analytics.py...')
r = req_dl.get(ANALYTICS_URL, timeout=30); r.raise_for_status()
with open('/content/analytics.py', 'w', encoding='utf-8') as f:
    f.write(r.text)
print('OK analytics.py descargado')

# Arrancar Ollama y descargar nomic-embed-text (default rapido para Colab gratis)
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print('Descargando modelo de embeddings (nomic-embed-text, ~270 MB)...')
subprocess.run(['ollama', 'pull', 'nomic-embed-text'], check=True)
print('OK Ollama listo con nomic-embed-text')


In [ ]:
# Celda 3: Servidor Flask (proxy LLM + endpoints paso 7 + estaticos)
import sys, os, threading, time, subprocess
sys.path.insert(0, '/content')

from flask import Flask, request, jsonify, send_from_directory
import requests as req
from analytics import handle  # router de acciones del paso 7

app = Flask(__name__)
MODEL_NAME = MODEL + ':cloud'

# ─── Helpers CORS ────────────────────────────────────────────────────────────
def _cors_preflight():
    resp = app.make_response('')
    resp.headers['Access-Control-Allow-Origin'] = '*'
    resp.headers['Access-Control-Allow-Headers'] = 'Content-Type,Authorization,X-Proxy-Auth'
    resp.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return resp

def _cors_json(data, status=200):
    resp = jsonify(data)
    resp.status_code = status
    resp.headers['Access-Control-Allow-Origin'] = '*'
    return resp

# ─── Config y chat ────────────────────────────────────────────────────────────
@app.route('/api/config')
def config():
    return jsonify({'api_url': API_URL, 'api_key': API_KEY, 'model': MODEL})

@app.route('/api/tags')
def tags():
    return jsonify({'models': [{'name': MODEL_NAME}]})

@app.route('/api/chat', methods=['POST', 'OPTIONS'])
def chat():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    data = request.json or {}
    messages = data.get('messages', [])
    auth_header = request.headers.get('X-Proxy-Auth') or request.headers.get('Authorization') or ('Bearer ' + API_KEY)
    body = {'model': MODEL, 'messages': messages, 'max_tokens': 8192}
    try:
        r = req.post(API_URL, headers={'Authorization': auth_header, 'Content-Type': 'application/json'}, json=body, timeout=300)
        r.encoding = 'utf-8'
        result = r.json()
        if 'error' in result:
            return _cors_json({'error': result['error']}, 500)
        content = result.get('choices', [{}])[0].get('message', {}).get('content', '')
        return _cors_json({'message': {'content': content}})
    except Exception as e:
        return _cors_json({'error': str(e)}, 502)

@app.route('/proxy', methods=['GET', 'POST', 'OPTIONS'])
def proxy():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    target_url = request.args.get('url', '')
    if not target_url:
        return _cors_json({'error': 'url param required'}, 400)
    auth_header = request.headers.get('X-Proxy-Auth') or request.headers.get('Authorization') or ('Bearer ' + API_KEY)
    headers = {'Authorization': auth_header, 'Content-Type': 'application/json'}
    try:
        if request.method == 'POST':
            body = {**request.json} if request.json else {}
            body['max_tokens'] = 8192
            if 'moonshot.ai' in target_url:
                body.pop('temperature', None)
            r = req.post(target_url, headers=headers, json=body, timeout=300)
        else:
            r = req.get(target_url, headers=headers, timeout=30)
        upstream_ct = r.headers.get('Content-Type', 'application/json')
        if 'charset' not in upstream_ct.lower():
            upstream_ct = upstream_ct.split(';')[0].strip() + '; charset=utf-8'
        resp = app.make_response(r.content)
        resp.status_code = r.status_code
        resp.headers['Content-Type'] = upstream_ct
        resp.headers['Access-Control-Allow-Origin'] = '*'
        return resp
    except Exception as e:
        return _cors_json({'error': str(e)}, 502)

# ─── Paso 7: /api/analytics ───────────────────────────────────────────────────
@app.route('/api/analytics', methods=['POST', 'OPTIONS'])
def api_analytics():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    payload = request.json or {}
    try:
        result = handle(payload.get('action'), payload.get('data', {}))
        return _cors_json(result)
    except Exception as e:
        return _cors_json({'error': str(e)}, 500)

# ─── Paso 7: embeddings via Ollama local con lazy pull ───────────────────────
_pulled_models = {'nomic-embed-text'}

@app.route('/api/analytics/embed', methods=['POST', 'OPTIONS'])
def api_embed():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    data = request.json or {}
    texts = data.get('texts', [])
    model = data.get('model', 'nomic-embed-text')
    if model not in _pulled_models:
        try:
            subprocess.run(['ollama', 'pull', model], check=True, timeout=900)
            _pulled_models.add(model)
        except Exception as e:
            return _cors_json({'error': f'No se pudo descargar {model}: {e}'}, 500)
    try:
        embeddings = []
        for t in texts:
            r = req.post('http://localhost:11434/api/embeddings',
                         json={'model': model, 'prompt': t}, timeout=120)
            embeddings.append(r.json().get('embedding', []))
        return _cors_json({'embeddings': embeddings})
    except Exception as e:
        return _cors_json({'error': str(e)}, 502)

# ─── Paso 7: sentimiento, normalize, narrate ─────────────────────────────────
def _llm_call(system, user, temperature=0.0, max_tokens=2048):
    body = {
        'model': MODEL,
        'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    r = req.post(API_URL, headers={'Authorization': 'Bearer ' + API_KEY, 'Content-Type': 'application/json'}, json=body, timeout=300)
    r.raise_for_status()
    r.encoding = 'utf-8'
    return r.json()['choices'][0]['message']['content']

@app.route('/api/analytics/sentiment', methods=['POST', 'OPTIONS'])
def api_sentiment():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    import json as _json
    data = request.json or {}
    texts = data.get('texts', [])
    lang_context = data.get('lang_context', '')
    BATCH = 40
    sentiments = []
    system = (
        'Eres un clasificador de sentimiento para respuestas de encuesta. ' + lang_context + '\n'
        'Dado un array JSON de respuestas, devuelve un array JSON del mismo tamano donde cada elemento es exactamente uno de: "positivo", "negativo", "ambivalente", "neutro".\n'
        'Responde SOLO el array JSON, sin texto adicional ni markdown.'
    )
    try:
        for i in range(0, len(texts), BATCH):
            batch = texts[i:i+BATCH]
            raw = _llm_call(system, _json.dumps(batch, ensure_ascii=False), temperature=0.0, max_tokens=1024)
            try:
                parsed = _json.loads(raw.replace('```json', '').replace('```', '').strip())
                sentiments.extend(parsed if isinstance(parsed, list) else ['neutro'] * len(batch))
            except Exception:
                sentiments.extend(['neutro'] * len(batch))
        return _cors_json({'sentiments': sentiments})
    except Exception as e:
        return _cors_json({'error': str(e)}, 502)

@app.route('/api/analytics/normalize', methods=['POST', 'OPTIONS'])
def api_normalize():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    import json as _json
    data = request.json or {}
    texts = data.get('texts', [])
    lang_context = data.get('lang_context', '')
    BATCH = 40
    normalized = []
    system = (
        'Eres un normalizador de texto para encuestas. ' + lang_context + '\n'
        'Dado un array JSON de respuestas, devuelve un array JSON del mismo tamano con cada texto normalizado '
        '(minusculas, ortografia corregida, abreviaciones expandidas, sin puntuacion innecesaria). '
        'Responde SOLO el array JSON.'
    )
    try:
        for i in range(0, len(texts), BATCH):
            batch = texts[i:i+BATCH]
            raw = _llm_call(system, _json.dumps(batch, ensure_ascii=False), temperature=0.0, max_tokens=4096)
            try:
                parsed = _json.loads(raw.replace('```json', '').replace('```', '').strip())
                normalized.extend(parsed if isinstance(parsed, list) and len(parsed) == len(batch) else batch)
            except Exception:
                normalized.extend(batch)
        return _cors_json({'normalized': normalized})
    except Exception as e:
        return _cors_json({'error': str(e)}, 502)

@app.route('/api/analytics/narrate', methods=['POST', 'OPTIONS'])
def api_narrate():
    if request.method == 'OPTIONS':
        return _cors_preflight()
    import json as _json
    data = request.json or {}
    clusters = data.get('clusters', [])
    pipeline = data.get('pipeline', 'A')
    lang_context = data.get('lang_context', '')
    system = (
        'Eres un analista cualitativo. ' + lang_context + '\n'
        f'Te paso una lista de clusters del Pipeline {pipeline}. Para cada uno te doy textos representativos y palabras clave.\n'
        'Devuelve un array JSON con un objeto por cluster (mismo orden) con los campos:\n'
        '- nombre: 2-5 palabras que sintetizan el cluster\n'
        '- descripcion: una frase de 1-2 lineas\n'
        '- tono: "positivo" | "negativo" | "ambivalente" | "neutro"\n'
        'Responde SOLO el array JSON.'
    )
    try:
        raw = _llm_call(system, _json.dumps(clusters, ensure_ascii=False), temperature=0.2, max_tokens=4096)
        try:
            parsed = _json.loads(raw.replace('```json', '').replace('```', '').strip())
        except Exception:
            parsed = []
        return _cors_json({'result': parsed})
    except Exception as e:
        return _cors_json({'error': str(e)}, 502)

# ─── Estaticos ────────────────────────────────────────────────────────────────
@app.route('/', defaults={'path': ''})
@app.route('/<path:path>')
def static_files(path):
    full = os.path.join(DIST_DIR, path)
    if path and os.path.isfile(full):
        return send_from_directory(DIST_DIR, path)
    return send_from_directory(DIST_DIR, 'index.html')

threading.Thread(target=lambda: app.run(port=PORT, host='0.0.0.0', use_reloader=False), daemon=True).start()

if not os.path.isfile(os.path.join(DIST_DIR, 'index.html')):
    raise RuntimeError('ERROR: ' + DIST_DIR + '/index.html no existe. Re-ejecuta Celda 2.')

flask_ok = False
for _ in range(20):
    time.sleep(0.5)
    try:
        r = req.get('http://localhost:' + str(PORT) + '/api/config', timeout=2)
        if r.status_code == 200:
            flask_ok = True
            break
    except Exception:
        pass

if not flask_ok:
    raise RuntimeError('ERROR: Flask no respondio en 10s.')

print('OK Servidor corriendo en puerto ' + str(PORT) + ' (Flask + paso 7 + Ollama)')


In [ ]:
# Celda 4: Cloudflare tunnel
import subprocess, threading, re, time

tunnel_url = None
tunnel_log = []

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:' + str(PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        tunnel_log.append(line.rstrip())
        if tunnel_url is None:
            m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
            if m:
                tunnel_url = m.group(1)
                print('\n+------------------------------------------------------+')
                print('  OK Abre esta URL en tu browser:')
                print('  ' + tunnel_url)
                print('+------------------------------------------------------+')
                print('  (No cierres esta pestana mientras usas la herramienta)')

threading.Thread(target=run_tunnel, daemon=True).start()

# Esperar hasta 60s a que aparezca la URL
for intento in range(60):
    if tunnel_url:
        break
    time.sleep(1)

if not tunnel_url:
    print('ERROR: cloudflared no genero URL en 60s.')
    print('Ultimas lineas del log:')
    for linea in tunnel_log[-20:]:
        print('  ' + linea)
    raise RuntimeError('Tunnel no establecido. Re-ejecuta esta celda o revisa el log.')


In [ ]:
# Celda 5: Keep-alive (corre indefinidamente, es normal)
import time
print('Sesion activa. La herramienta esta disponible en la URL de arriba.')
print('Para terminar: Entorno de ejecucion -> Interrumpir ejecucion')
i = 0
while True:
    time.sleep(60)
    i += 1
    print('[' + str(i) + ' min] activo', end='\r')